Creditos de la versión Inicial: Juan Felipe Fajardo Garzón


In [1]:
!pip install wordfreq
!pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [11]:
from IPython.display import Audio, display
from gtts import gTTS
import difflib
import random

# ==========================================
# 1. CONFIGURACIÓN DE PARÁMETROS
# ==========================================
PALABRA_OBJETIVO = "unicornio"
N_POBLACION = 100
PROB_MUTACION = 0.18
GENERACIONES = 500
ALFABETO = list("abcdefghijklmnopqrstuvwxyzñ")

# ==========================================
# 2. FUNCIONES DEL ALGORITMO GENÉTICO
# ==========================================
def generar_poblacion_inicial():
    """Genera una población inicial con palabras aleatorias."""
    poblacion = []
    for _ in range(N_POBLACION):
        palabra = "".join(random.choices(ALFABETO, k=random.randint(3, 8)))
        poblacion.append(palabra)
    return poblacion

def aptitud(palabra):
    """Calcula la similitud de una palabra con el objetivo."""
    return difflib.SequenceMatcher(None, palabra, PALABRA_OBJETIVO).ratio()

def seleccion(poblacion_evaluada):
    """Selecciona 2 padres usando el método de la ruleta."""
    palabras = [ind[0] for ind in poblacion_evaluada]
    pesos = [ind[1] for ind in poblacion_evaluada]

    if sum(pesos) == 0:
        pesos = [1] * len(palabras)

    return random.choices(palabras, weights=pesos, k=2)

def cruce(padre1, padre2):
    """Cruza dos palabras cortando en un punto aleatorio."""
    if not padre1 or not padre2:
        return padre1 or padre2 or random.choice(ALFABETO)

    punto_corte = random.randint(0, min(len(padre1), len(padre2)))
    return padre1[:punto_corte] + padre2[punto_corte:]

def mutacion(palabra):
    """Aplica mutación inteligente: puede sustituir, insertar o eliminar."""
    if not palabra:
        return random.choice(ALFABETO)

    if random.random() <= PROB_MUTACION:
        caracteres = list(palabra)
        tipo = random.choice(["sustituir", "insertar", "eliminar"])

        if tipo == "eliminar" and len(caracteres) > 1:
            idx = random.randint(0, len(caracteres) - 1)
            caracteres.pop(idx)
        elif tipo == "insertar":
            idx = random.randint(0, len(caracteres))
            caracteres.insert(idx, random.choice(ALFABETO))
        else:
            idx = random.randint(0, len(caracteres) - 1)
            caracteres[idx] = random.choice(ALFABETO)

        return "".join(caracteres)

    return palabra

def reproducir_voz(poblacion, nombre_archivo="voz.mp3"):
    """Genera y reproduce por voz las palabras de la población o palabra."""
    try:
        texto = " ".join(poblacion)
        tts = gTTS(text=texto, lang="es")
        tts.save(nombre_archivo)
        display(Audio(nombre_archivo, autoplay=False))
    except Exception as e:
        print("Aviso: No se pudo generar la voz.", e)

# ==========================================
# 3. EJECUCIÓN PRINCIPAL
# ==========================================
print("--- Iniciando Algoritmo Genético ---")
poblacion = generar_poblacion_inicial()
# Se omite reproducir la voz inicial para no saturar de audios,
# pero puedes habilitarlo llamando a reproducir_voz(poblacion, "inicial.mp3")

print(f"\nPalabra objetivo: '{PALABRA_OBJETIVO}'\n")

# Variables para guardar al ganador absoluto
mejor_palabra_global = ""
mejor_aptitud_global = 0.0

for generacion in range(GENERACIONES):
    poblacion_evaluada = [(palabra, aptitud(palabra)) for palabra in poblacion]
    poblacion_evaluada.sort(key=lambda x: x[1], reverse=True)
    mejor_palabra, mejor_aptitud = poblacion_evaluada[0]

    mejor_palabra_global = mejor_palabra
    mejor_aptitud_global = mejor_aptitud

    print(f"Generación {generacion} | Mejor: '{mejor_palabra}' | Similitud: {mejor_aptitud * 100:.2f}%")

    if mejor_aptitud == 1.0:
        print(f"\n¡Éxito! Palabra objetivo '{PALABRA_OBJETIVO}' encontrada en la generación {generacion}.")
        break

    nueva_poblacion = [ind[0] for ind in poblacion_evaluada[:5]]

    while len(nueva_poblacion) < N_POBLACION:
        padre1, padre2 = seleccion(poblacion_evaluada)
        hijo = cruce(padre1, padre2)
        hijo = mutacion(hijo)
        nueva_poblacion.append(hijo)

    poblacion = nueva_poblacion

# ==========================================
# 4. RESULTADOS Y AUDIOS FINALES
# ==========================================
print("\n--- Audio 1: Población Final Completa ---")
reproducir_voz(poblacion, "poblacion_final.mp3")

print("\n--- Audio 2: Cromosoma Ganador ---")
print(f"Mejor cromosoma logrado: '{mejor_palabra_global}' ({mejor_aptitud_global * 100:.2f}%)")
reproducir_voz([mejor_palabra_global], "ganador.mp3")

--- Iniciando Algoritmo Genético ---

Palabra objetivo: 'unicornio'

Generación 0 | Mejor: 'icni' | Similitud: 61.54%
Generación 1 | Mejor: 'icni' | Similitud: 61.54%
Generación 2 | Mejor: 'icni' | Similitud: 61.54%
Generación 3 | Mejor: 'nicni' | Similitud: 71.43%
Generación 4 | Mejor: 'nicni' | Similitud: 71.43%
Generación 5 | Mejor: 'nicni' | Similitud: 71.43%
Generación 6 | Mejor: 'nicni' | Similitud: 71.43%
Generación 7 | Mejor: 'nicni' | Similitud: 71.43%
Generación 8 | Mejor: 'uicniol' | Similitud: 75.00%
Generación 9 | Mejor: 'uicniol' | Similitud: 75.00%
Generación 10 | Mejor: 'uicniol' | Similitud: 75.00%
Generación 11 | Mejor: 'uicniol' | Similitud: 75.00%
Generación 12 | Mejor: 'uicniol' | Similitud: 75.00%
Generación 13 | Mejor: 'uicniol' | Similitud: 75.00%
Generación 14 | Mejor: 'niconiol' | Similitud: 82.35%
Generación 15 | Mejor: 'niconiol' | Similitud: 82.35%
Generación 16 | Mejor: 'niconiol' | Similitud: 82.35%
Generación 17 | Mejor: 'niconio' | Similitud: 87.50%
Gen


--- Audio 2: Cromosoma Ganador ---
Mejor cromosoma logrado: 'unicornio' (100.00%)
